In [2]:
import gymnasium as gym
import numpy as np
from bsk_rl import act, data, obs, scene, sats
from bsk_rl.sim import dyn, fsw

from Basilisk.architecture import bskLogging

bskLogging.setDefaultLogLevel(bskLogging.BSK_WARNING)


In [3]:
class MyScanningSatellite(sats.AccessSatellite):
    observation_spec = [
        obs.SatProperties(
            dict(prop="storage_level_fraction"),
            dict(prop="battery_charge_fraction")
        ),
        obs.Eclipse(),
    ]
    action_spec = [
        act.Scan(duration=60.0),
        act.Charge(duration=600.0),
    ]
    dyn_type = dyn.ContinuousImagingDynModel
    fsw_type = fsw.ContinuousImagingFSWModel

In [ ]:
MyScanningSatellite.default_sat_args()

{'hs_min': 0.0,
 'maxCounterValue': 4,
 'thrMinFireTime': 0.02,
 'desatAttitude': 'sun',
 'controlAxes_B': [1, 0, 0, 0, 1, 0, 0, 0, 1],
 'thrForceSign': 1,
 'K': 7.0,
 'Ki': -1,
 'P': 35.0,
 'imageAttErrorRequirement': 0.01,
 'imageRateErrorRequirement': None,
 'inst_pHat_B': [0, 0, 1],
 'utc_init': 'this value will be set by the world model',
 'batteryStorageCapacity': 288000.0,
 'storedCharge_Init': <function bsk_rl.sim.dyn.base.BasicDynamicsModel.<lambda>()>,
 'disturbance_vector': None,
 'dragCoeff': 2.2,
 'imageTargetMaximumRange': -1,
 'instrumentBaudRate': 8000000.0,
 'instrumentPowerDraw': -30.0,
 'basePowerDraw': 0.0,
 'wheelSpeeds': <function bsk_rl.sim.dyn.base.BasicDynamicsModel.<lambda>()>,
 'maxWheelSpeed': inf,
 'u_max': 0.2,
 'rwBasePower': 0.4,
 'rwMechToElecEfficiency': 0.0,
 'rwElecToMechEfficiency': 0.5,
 'panelArea': 1.0,
 'panelEfficiency': 0.2,
 'nHat_B': array([ 0,  0, -1]),
 'mass': 330,
 'width': 1.38,
 'depth': 1.04,
 'height': 1.58,
 'sigma_init': <function 

In [7]:
sat_args = {}

sat_args["imageAttErrorRequirement"] = 0.05
sat_args["dataStorageCapacity"] = 1e10
sat_args["instrumentBaudRate"] = 1e7
sat_args["storedCharge_Init"] = 50000.0

sat_args["storageInit"] = lambda: np.random.uniform(0.25, 0.75) * 1e10

sat = MyScanningSatellite(name="EO1", sat_args=sat_args)

Making the Environment


In [8]:
env = gym.make(
    "SatelliteTasking-v1",
    satellite = sat, # 配置的卫星
    scenario = scene.UniformNadirScanning(), # 定义的场景
    rewarder=data.ScanningTimeReward(),
    time_limit=5700.0,
    log_level="INFO",
)

2025-12-06 11:15:23,024 gym                            INFO       Calling env.reset() to get observation space
2025-12-06 11:15:23,025 gym                            INFO       Resetting environment with seed=2796862488
2025-12-06 11:15:23,140 sats.satellite.EO1             INFO       <0.00> EO1: Finding opportunity windows from 0.00 to 6000.00 seconds
2025-12-06 11:15:23,150 gym                            INFO       <0.00> Environment reset


Interaction With Env

In [9]:
observation, info = env.reset(seed=1)

2025-12-06 11:17:32,140 gym                            INFO       Resetting environment with seed=1
2025-12-06 11:17:32,294 sats.satellite.EO1             INFO       <0.00> EO1: Finding opportunity windows from 0.00 to 6000.00 seconds
2025-12-06 11:17:32,303 gym                            INFO       <0.00> Environment reset


In [10]:
print("Initial data level:", observation[0], "(randomized by sat_args)")
for _ in range(3):
    observation, reward, terminated, truncated, info = env.step(action=0)
print("  Final data level:", observation[0])

2025-12-06 11:20:21,664 gym                            INFO       <0.00> === STARTING STEP ===
2025-12-06 11:20:21,664 sats.satellite.EO1             INFO       <0.00> EO1: action_nadir_scan tasked for 60.0 seconds
2025-12-06 11:20:21,665 sats.satellite.EO1             INFO       <0.00> EO1: setting timed terminal event at 60.0
2025-12-06 11:20:21,675 sats.satellite.EO1             INFO       <60.00> EO1: timed termination at 60.0 for action_nadir_scan
2025-12-06 11:20:21,677 data.base                      INFO       <60.00> Total reward: {}
2025-12-06 11:20:21,678 comm.communication             INFO       <60.00> Optimizing data communication between all pairs of satellites
2025-12-06 11:20:21,679 sats.satellite.EO1             INFO       <60.00> EO1: Satellite EO1 requires retasking
2025-12-06 11:20:21,680 gym                            INFO       <60.00> Step reward: 0.0
2025-12-06 11:20:21,681 gym                            INFO       <60.00> === STARTING STEP ===
2025-12-06 11:20:

Initial data level: 0.5961613078 (randomized by sat_args)
  Final data level: 0.6741613078


In [11]:
while not truncated:
    observation, reward, terminated, truncated, info = env.step(action=1)
    print(f"Charge level: {observation[1]:.3f} ({env.unwrapped.simulator.sim_time:.1f} seconds)\n\tEclipse: start: {observation[2]:.1f} end: {observation[3]:.1f}")

2025-12-08 12:57:55,982 gym                            INFO       <180.00> === STARTING STEP ===
2025-12-08 12:57:55,994 sats.satellite.EO1             INFO       <180.00> EO1: action_charge tasked for 600.0 seconds
2025-12-08 12:57:55,995 sats.satellite.EO1             INFO       <180.00> EO1: setting timed terminal event at 780.0
2025-12-08 12:57:56,068 sats.satellite.EO1             INFO       <780.00> EO1: timed termination at 780.0 for action_charge
2025-12-08 12:57:56,072 data.base                      INFO       <780.00> Total reward: {}
2025-12-08 12:57:56,075 comm.communication             INFO       <780.00> Optimizing data communication between all pairs of satellites
2025-12-08 12:57:56,076 sats.satellite.EO1             INFO       <780.00> EO1: Satellite EO1 requires retasking
2025-12-08 12:57:56,094 gym                            INFO       <780.00> Step reward: 0.0
2025-12-08 12:57:56,095 gym                            INFO       <780.00> === STARTING STEP ===
2025-12-08

Charge level: 0.637 (780.0 seconds)
	Eclipse: start: 5610.0 end: 2040.0
Charge level: 0.635 (1380.0 seconds)
	Eclipse: start: 5010.0 end: 1440.0
Charge level: 0.632 (1980.0 seconds)
	Eclipse: start: 4410.0 end: 840.0
Charge level: 0.630 (2580.0 seconds)
	Eclipse: start: 3810.0 end: 240.0
Charge level: 1.000 (3180.0 seconds)
	Eclipse: start: 3210.0 end: 5310.0
Charge level: 1.000 (3780.0 seconds)
	Eclipse: start: 2610.0 end: 4710.0


2025-12-08 12:57:56,309 sats.satellite.EO1             INFO       <4380.00> EO1: timed termination at 4380.0 for action_charge
2025-12-08 12:57:56,310 data.base                      INFO       <4380.00> Total reward: {}
2025-12-08 12:57:56,311 comm.communication             INFO       <4380.00> Optimizing data communication between all pairs of satellites
2025-12-08 12:57:56,311 sats.satellite.EO1             INFO       <4380.00> EO1: Satellite EO1 requires retasking
2025-12-08 12:57:56,313 gym                            INFO       <4380.00> Step reward: 0.0
2025-12-08 12:57:56,313 gym                            INFO       <4380.00> === STARTING STEP ===
2025-12-08 12:57:56,313 sats.satellite.EO1             INFO       <4380.00> EO1: action_charge tasked for 600.0 seconds
2025-12-08 12:57:56,313 sats.satellite.EO1             INFO       <4380.00> EO1: setting timed terminal event at 4980.0
2025-12-08 12:57:56,343 sats.satellite.EO1             INFO       <4980.00> EO1: timed terminatio

Charge level: 1.000 (4380.0 seconds)
	Eclipse: start: 2010.0 end: 4110.0
Charge level: 1.000 (4980.0 seconds)
	Eclipse: start: 1410.0 end: 3510.0
Charge level: 1.000 (5580.0 seconds)
	Eclipse: start: 810.0 end: 2910.0
Charge level: 1.000 (5700.0 seconds)
	Eclipse: start: 690.0 end: 2790.0


In [12]:
!conda list

# packages in environment at D:\miniconda\envs\Basilisk:
#
# Name                    Version                   Build  Channel
alabaster                 1.0.0                    pypi_0    pypi
anyio                     4.12.0                   pypi_0    pypi
argon2-cffi               25.1.0                   pypi_0    pypi
argon2-cffi-bindings      25.1.0                   pypi_0    pypi
arrow                     1.4.0                    pypi_0    pypi
asttokens                 3.0.1                    pypi_0    pypi
attrs                     25.4.0                   pypi_0    pypi
babel                     2.17.0                   pypi_0    pypi
beautifulsoup4            4.14.3                   pypi_0    pypi
bleach                    6.3.0                    pypi_0    pypi
bsk-rl                    1.2.13                   pypi_0    pypi
bzip2                     1.0.8                h2bbff1b_6    https://mirrors.tuna.tsinghua.edu.cn/anaconda/pkgs/main
ca-certificates           2025.